# Sous-titrage arabe → français : piste EmbeddingGemma

Le groupe compare plusieurs systèmes de traduction directe de journaux télévisés arabes vers le français : S0 (Whisper-small + Opus-MT), S1/S2 (cascade Whisper + Qwen3-4B LoRA), S3 (pont audio-LLM) et le mode direct à 2-3 s de latence, mesurés en chrF++, BLEU, COMET, WER et RTF. Ma part est de tester google/embeddinggemma-300m.

EmbeddingGemma est un encodeur (un vecteur de 768 dimensions par texte), il ne génère pas de texte. On ne peut donc pas le comparer directement à S0-S3, alors je le teste là où il pourrait servir dans le pipeline :
- A. estimation de qualité (QE), pour remplacer ou compléter COMETKiwi au filtre D4-7 (section 6)
- B. mémoire de traduction, récupérer une traduction au lieu d'en générer une (section 7)
- C. détection des noms propres du lexique de S2 (section 8)
- D. retrieval cross-lingue AR/FR, comme diagnostic (section 5)

Données : transcription.xlsx, 227 segments arabes d'un épisode d'environ 49 min, sans traduction française. Les traductions candidates sont donc produites avec Opus-MT et NLLB. Avec le jeu T/R du groupe (colonnes ar et fr), on met son chemin dans T_PATH et on obtient de vrais scores avec références. EXTERNAL_HYP permet d'ajouter les sorties d'autres systèmes (S2, S3...).

Avant de lancer : runtime GPU (T4), licence Gemma acceptée, secret HF_TOKEN dans Colab.

In [ ]:
!pip install -q -U "sentence-transformers>=5.0" "transformers>=4.56" accelerate sentencepiece sacrebleu openpyxl scikit-learn scipy matplotlib

In [ ]:
import os, sys, re, time, random, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
IN_COLAB = "google.colab" in sys.modules
print("device :", device)

# paramètres
DATA_PATH   = "transcription.xlsx"      # arabe seul (colonne 'Transcription')
T_PATH      = None                      # jeu T/R du groupe, colonnes ar, fr (+ id)
BASE_PATH   = None                      # base parallèle (ar, fr) pour la mémoire de traduction (jeu D/P)
EMB_MODEL   = "google/embeddinggemma-300m"
MT_SYSTEMS  = {
    "opus-mt-ar-fr (S0)": "Helsinki-NLP/opus-mt-ar-fr",
    "nllb-200-600M":      "facebook/nllb-200-distilled-600M",
}
EXTERNAL_HYP = {}                       # ex. {"S2": "hyp_s2.csv"}, colonnes uid, fr
LEXIQUE_URL = "https://raw.githubusercontent.com/SamiNakibETU/NLP/main/donnees/noms/noms_v1.csv"
CHUNK_MAX_WORDS = 40                    # None = garder les segments d'origine
MIN_WORDS       = 3
RUN_BASELINES   = True                  # LaBSE, mpnet, e5
RUN_COMETKIWI   = False                 # à lancer en dernier (installe unbabel-comet)

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()

## 1. Données
Deux sources possibles, avec le même format de sortie D (colonnes uid, ar, ref) :
- le jeu T/R du groupe (T_PATH), avec la traduction humaine dans ref
- sinon transcription.xlsx, arabe seul (ref vide), avec les longs segments découpés en chunks d'environ 40 mots parce que le live sous-titre par petits morceaux

In [ ]:
DIACRITICS = re.compile(r"[\u064B-\u0652\u0670]")
def clean_ar(t):
    t = str(t).replace("\u0640", "")
    t = DIACRITICS.sub("", t)
    t = re.sub(r"\s+([،\.؟\?!؛;:,])", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

def hard_split(words, max_words):
    n = max(1, round(len(words) / max_words)); size = -(-len(words) // n)
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

def split_text(text, max_words):
    parts = [p.strip() for p in re.split(r"(?<=[،\.؟\?!؛;])\s+", text) if p.strip()]
    flat = []
    for p in parts:
        w = p.split()
        flat += hard_split(w, max_words) if len(w) > max_words else [p]
    chunks, cur = [], []
    for p in flat:
        if cur and len(" ".join(cur + [p]).split()) > max_words:
            chunks.append(" ".join(cur)); cur = [p]
        else:
            cur.append(p)
    if cur: chunks.append(" ".join(cur))
    return chunks

def read_table(path):
    return pd.read_json(path, lines=True) if str(path).endswith(".jsonl") else pd.read_csv(path)

if T_PATH and os.path.exists(T_PATH):
    t = read_table(T_PATH)
    D = pd.DataFrame({"uid": t["id"].astype(str) if "id" in t else t.index.astype(str),
                      "ar": t["ar"].map(clean_ar), "ref": t["fr"].astype(str)})
    SOURCE = f"jeu du groupe ({T_PATH})"
else:
    if not os.path.exists(DATA_PATH) and IN_COLAB:
        from google.colab import files
        DATA_PATH = list(files.upload().keys())[0]
    df = pd.read_excel(DATA_PATH).rename(columns={"Transcription": "ar", "Début": "debut", "Fin": "fin"})
    rows = []
    for sid, txt in enumerate(df["ar"]):
        pieces = split_text(clean_ar(txt), CHUNK_MAX_WORDS) if CHUNK_MAX_WORDS else [clean_ar(txt)]
        for k, p in enumerate(pieces):
            if len(p.split()) >= MIN_WORDS:
                rows.append({"uid": f"{sid}-{k}", "ar": p, "ref": np.nan})
    D = pd.DataFrame(rows)
    SOURCE = f"{DATA_PATH} (arabe seul)"

D["nb_mots"] = D["ar"].str.split().str.len()
print("Source :", SOURCE, "|", len(D), "unités | mots/unité : médiane", int(D.nb_mots.median()), ", max", D.nb_mots.max())
D[["uid", "ar"]].to_csv("segments_pour_le_groupe.csv", index=False)   # pour les autres systèmes du groupe
D.head()

## 2. Traductions candidates
Chaque système produit une traduction par segment, qui sert pour la QE de la section 6. Opus-MT et NLLB sont générés ici, les systèmes du groupe s'ajoutent avec EXTERNAL_HYP (csv avec uid et fr). Le fichier segments_pour_le_groupe.csv donne les uid et l'arabe à traduire.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(path, texts, bs=16, num_beams=4, max_new_tokens=256):
    is_nllb = "nllb" in path
    tok = AutoTokenizer.from_pretrained(path, **({"src_lang": "arb_Arab"} if is_nllb else {}))
    mt = AutoModelForSeq2SeqLM.from_pretrained(path).to(device).eval()
    kw = dict(num_beams=num_beams, max_new_tokens=max_new_tokens)
    if is_nllb: kw["forced_bos_token_id"] = tok.convert_tokens_to_ids("fra_Latn")
    texts = list(texts); order = np.argsort([len(t) for t in texts])[::-1]; out = [None] * len(texts)
    for i in tqdm(range(0, len(texts), bs), desc=path.split("/")[-1]):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            gen = mt.generate(**enc, **kw)
        for j, d in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = d
    del mt; gc.collect(); torch.cuda.empty_cache()
    return out

for name, path in MT_SYSTEMS.items():
    D[f"hyp::{name}"] = translate(path, D.ar)

for name, path in EXTERNAL_HYP.items():
    ext = pd.read_csv(path)[["uid", "fr"]].rename(columns={"fr": f"hyp::{name}"})
    ext["uid"] = ext["uid"].astype(str)
    D = D.merge(ext, on="uid", how="left")

HYPS = [c for c in D.columns if c.startswith("hyp::")]
HAS_REF = D["ref"].notna().all()
FR_PROXY = HYPS[0]
D["fr"] = D["ref"] if HAS_REF else D[FR_PROXY]
print("Systèmes :", [h[5:] for h in HYPS])
print("Référence humaine :", "oui" if HAS_REF else f"non -> le français de référence = sortie de {FR_PROXY[5:]} (scores optimistes)")
for i in [0, 5]:
    print("AR :", D.ar[i][:140]); [print(f"{h[5:]:>22} :", D[h][i][:140]) for h in HYPS]; print("-" * 40)
D.to_csv("hypotheses.csv", index=False)

In [ ]:
from sacrebleu.metrics import CHRF, BLEU
chrfpp = CHRF(word_order=2)          # chrF++ (comme dans le dépôt du groupe)
bleu = BLEU()

def seg_chrf(hyps, refs):
    return np.array([chrfpp.sentence_score(h, [r]).score for h, r in zip(hyps, refs)])

if HAS_REF:
    rows = [{"système": h[5:], "chrF++": round(chrfpp.corpus_score(D[h].tolist(), [D.ref.tolist()]).score, 1),
             "BLEU": round(bleu.corpus_score(D[h].tolist(), [D.ref.tolist()]).score, 1)} for h in HYPS]
    res_mt = pd.DataFrame(rows); display(res_mt)
else:
    print("Pas de référence humaine : on ne peut pas noter les systèmes en chrF++ ici. La QE (section 6) sert justement à ça.")
    res_mt = pd.DataFrame()

## 3-4. EmbeddingGemma et outils d'évaluation
Prompts : `task: search result | query: ` pour la requête, `title: none | text: ` pour le document, `task: sentence similarity | query: ` pour la similarité symétrique. Leur effet est mesuré en section 5. Pas de float16 (float32 ou bfloat16), et les vecteurs sont normalisés donc cosinus = produit scalaire.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMB_MODEL, device=device)
print("dimension :", model.get_sentence_embedding_dimension(), "| max_seq_length :", model.max_seq_length)

Q_PROMPT = "task: search result | query: "
D_PROMPT = "title: none | text: "
S_PROMPT = "task: sentence similarity | query: "

def embed(m, texts, prompt="", bs=16):
    return m.encode(list(texts), prompt=prompt or None, batch_size=bs, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=False)

def truncate(E, dim):
    E = E[:, :dim]; return E / np.linalg.norm(E, axis=1, keepdims=True)

def retrieval_metrics(sim, ks=(1, 5, 10), window=1):
    sim = np.asarray(sim); n = sim.shape[0]
    gold = sim[np.arange(n), np.arange(n)][:, None]
    ranks = (sim > gold).sum(1) + 1
    out = {f"R@{k}": float((ranks <= k).mean()) for k in ks}
    out["MRR"] = float((1.0 / ranks).mean()); out["rang médian"] = float(np.median(ranks))
    out[f"R@1 ±{window}"] = float((np.abs(sim.argmax(1) - np.arange(n)) <= window).mean())
    return out

def add_asr_noise(text, p, seed=0):
    # bruit de transcription : ponctuation retirée, mots perdus, deux lettres permutées
    rng = random.Random(seed); text = re.sub(r"[^\w\s']", " ", text)
    out = []
    for w in text.split():
        r = rng.random()
        if r < p / 2: continue
        if r < p and len(w) > 3:
            i = rng.randrange(len(w) - 1); w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        out.append(w)
    return " ".join(out) if out else text

a = embed(model, D.ar.iloc[:2], S_PROMPT); b = embed(model, D.fr.iloc[:2], S_PROMPT)
print("cos(ar0, fr0) =", round(float(a[0] @ b[0]), 3), "| cos(ar0, fr1) =", round(float(a[0] @ b[1]), 3))

## 5. Diagnostic cross-lingue
Pour chaque phrase arabe, on classe toutes les phrases françaises, la bonne étant celle de même indice. R@k : la bonne est dans le top k. MRR : moyenne de 1/rang. R@1 ±1 accepte le chunk voisin. Ce n'est pas une mesure de traduction, ça dit seulement si l'espace vectoriel aligne bien l'arabe et le français.

In [ ]:
configs = {
    "sans prompt":             dict(q="",       d=""),
    "STS (symétrique)":        dict(q=S_PROMPT, d=S_PROMPT),
    "Retrieval (asymétrique)": dict(q=Q_PROMPT, d=D_PROMPT),
}
rows, cache = [], {}
for name, c in configs.items():
    ar_q = embed(model, D.ar, c["q"]); fr_d = embed(model, D.fr, c["d"])
    fr_q = embed(model, D.fr, c["q"]); ar_d = embed(model, D.ar, c["d"])
    rows.append({"config": name, "direction": "AR→FR", **retrieval_metrics(ar_q @ fr_d.T)})
    rows.append({"config": name, "direction": "FR→AR", **retrieval_metrics(fr_q @ ar_d.T)})
    cache[name] = dict(ar_q=ar_q, fr_d=fr_d)
res1 = pd.DataFrame(rows).round(3); display(res1)

BEST = res1[res1.direction == "AR→FR"].sort_values("MRR").iloc[-1]["config"]
P = configs[BEST]; E_AR_Q, E_FR_D = cache[BEST]["ar_q"], cache[BEST]["fr_d"]
print("Meilleure config (AR→FR, MRR) :", BEST)

In [ ]:
# dimensions matryoshka, qualité vs taille d'index
rows = []
for dim in [768, 512, 256, 128]:
    q, d = truncate(E_AR_Q, dim), truncate(E_FR_D, dim)
    rows.append({"dim": dim, "taille index (Ko)": round(d.nbytes / 1024, 1), **retrieval_metrics(q @ d.T)})
res2 = pd.DataFrame(rows).round(3); display(res2)

In [ ]:
# bruit ASR sur l'arabe
rows = []
for name, p in [("propre", None), ("sans ponctuation", 0.0), ("p=0.05", 0.05), ("p=0.10", 0.10), ("p=0.20", 0.20), ("p=0.30", 0.30)]:
    ar_noisy = D.ar.tolist() if p is None else [add_asr_noise(t, p, SEED + i) for i, t in enumerate(D.ar)]
    rows.append({"bruit": name, **retrieval_metrics(embed(model, ar_noisy, P["q"]) @ E_FR_D.T)})
res3 = pd.DataFrame(rows).round(3); display(res3)

plt.figure(figsize=(6, 3.5))
for k, mk in [("R@1", "o"), ("R@5", "s"), ("MRR", "^")]:
    plt.plot(res3["bruit"], res3[k], marker=mk, label=k)
plt.xticks(rotation=30); plt.ylim(0, 1.02); plt.legend(); plt.title("Robustesse au bruit ASR (AR→FR)")
plt.tight_layout(); plt.show()

In [ ]:
# latence d'un encodage (budget du mode direct : 2-3 s)
def bench(fn, n=30):
    fn(); ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn()
        if device == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return round(float(np.median(ts)), 1), round(float(np.percentile(ts, 95)), 1)

s = D.ar.iloc[10]
single = bench(lambda: embed(model, [s], P["q"]))
batch  = bench(lambda: embed(model, D.ar.iloc[:32].tolist(), P["q"], bs=32), n=5)
print(f"1 segment : médiane {single[0]} ms | p95 {single[1]} ms")
print(f"32 segments : médiane {batch[0]} ms | p95 {batch[1]} ms")
res4 = pd.DataFrame([{"opération": "encoder 1 segment arabe", "ms (médiane)": single[0], "ms (p95)": single[1], "device": device}])

In [ ]:
# autres encodeurs, mêmes données
baselines = {
    "LaBSE": ("sentence-transformers/LaBSE", "", ""),
    "paraphrase-multilingual-mpnet": ("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "", ""),
    "multilingual-e5-base": ("intfloat/multilingual-e5-base", "query: ", "passage: "),
}
rows = [{"modèle": f"EmbeddingGemma ({BEST})", **retrieval_metrics(E_AR_Q @ E_FR_D.T)}]
if RUN_BASELINES:
    for name, (path, qp, dp) in baselines.items():
        try:
            m = SentenceTransformer(path, device=device)
            rows.append({"modèle": name, **retrieval_metrics(embed(m, D.ar, qp) @ embed(m, D.fr, dp).T)})
            del m; gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print("Échec", name, "->", e)
res5 = pd.DataFrame(rows).round(3); display(res5)

## 6. Rôle A : estimation de qualité sans référence
Le groupe filtre les données d'entraînement avec COMETKiwi (on retire les 20 % les moins bien notés). Ici le score est le cos(arabe, français candidat) d'EmbeddingGemma. Validation de quatre façons :
1. paires mélangées : un arabe avec la traduction d'un autre segment doit avoir un score plus bas (AUC)
2. dégradations : traduction tronquée à 50 %, 30 % de mots supprimés (AUC)
3. corrélation avec chrF++ par segment, si on a une référence humaine
4. accord avec COMETKiwi (section 9, optionnelle)

L'AUC est la probabilité qu'une bonne paire ait un score plus haut qu'une paire dégradée (0,5 = hasard, 1 = parfait).

In [ ]:
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

E_ar_s = embed(model, D.ar, S_PROMPT)
def qe_scores(hyp_texts):
    return (E_ar_s * embed(model, hyp_texts, S_PROMPT)).sum(1)

def corrupt_truncate(t, frac=0.5):
    w = t.split(); return " ".join(w[:max(1, int(len(w) * frac))])
def corrupt_dropout(t, p=0.3, seed=0):
    rng = random.Random(seed); w = [x for x in t.split() if rng.random() > p]
    return " ".join(w) if w else t

def auc(good, bad):
    return round(roc_auc_score(np.r_[np.ones(len(good)), np.zeros(len(bad))], np.r_[good, bad]), 3)

rng = np.random.default_rng(SEED)
rows, QE = [], {}
for h in HYPS:
    good = qe_scores(D[h].tolist()); QE[h] = good
    perm = rng.permutation(len(D)); perm = np.where(perm == np.arange(len(D)), (perm + 1) % len(D), perm)
    shuffled = qe_scores(D[h].values[perm].tolist())
    trunc = qe_scores([corrupt_truncate(t) for t in D[h]])
    drop = qe_scores([corrupt_dropout(t, 0.3, SEED + i) for i, t in enumerate(D[h])])
    row = {"système": h[5:], "cos moyen": round(float(good.mean()), 3),
           "AUC paires mélangées": auc(good, shuffled), "AUC tronquée 50%": auc(good, trunc), "AUC mots perdus 30%": auc(good, drop)}
    if HAS_REF:
        rho, _ = spearmanr(good, seg_chrf(D[h], D.ref)); row["Spearman(cos, chrF++)"] = round(float(rho), 3)
    rows.append(row)
res6 = pd.DataFrame(rows); display(res6)

In [ ]:
# filtre 7 : on retire les 20 % les moins bien notés (qe_rejet_pire_pct = 20)
rows = []
for h in HYPS:
    s = QE[h]; thr = np.quantile(s, 0.20); keep = s > thr
    row = {"système": h[5:], "seuil cos (20 % pires)": round(float(thr), 3), "gardés": int(keep.sum()), "rejetés": int((~keep).sum())}
    if HAS_REF:
        c = seg_chrf(D[h], D.ref); row["chrF++ moyen gardés"] = round(float(c[keep].mean()), 1); row["chrF++ moyen rejetés"] = round(float(c[~keep].mean()), 1)
    rows.append(row)
res6b = pd.DataFrame(rows); display(res6b)

h0 = HYPS[0]; order = np.argsort(QE[h0])
print(f"\n5 segments les plus mal notés pour {h0[5:]} :")
for i in order[:5]:
    print(f"[{QE[h0][i]:.2f}] AR : {D.ar[i][:90]}\n        FR : {D[h0][i][:90]}")

## 7. Rôle B : mémoire de traduction
Pour chaque segment arabe, on cherche l'arabe le plus proche dans une base parallèle et on renvoie sa traduction française. S0 à S3 génèrent, ici on recopie, donc ça ne marche que si la phrase (ou une phrase très proche) est dans la base.

La base est BASE_PATH (jeu D ou P, colonnes ar et fr) si fourni, sinon les 80 % premiers segments de l'épisode, avec test sur les 20 % derniers. Avec la seule base de l'épisode je m'attends à un chrF++ faible, sauf quand la similarité est très haute, d'où le calcul du chrF++ selon le seuil de similarité.

In [ ]:
if BASE_PATH and os.path.exists(BASE_PATH):
    bt = read_table(BASE_PATH); base_ar = bt["ar"].map(clean_ar).tolist(); base_fr = bt["fr"].astype(str).tolist(); test = D
    print("Base parallèle externe :", len(base_ar), "paires")
else:
    cut = int(0.8 * len(D)); base_ar, base_fr = D.ar.iloc[:cut].tolist(), D.fr.iloc[:cut].tolist(); test = D.iloc[cut:].reset_index(drop=True)
    print(f"Base interne : {len(base_ar)} paires | test : {len(test)} segments")

B = embed(model, base_ar, S_PROMPT); T = embed(model, test.ar, S_PROMPT)
S = T @ B.T; top = S.argmax(1); top_score = S.max(1)
tm_fr = [base_fr[j] for j in top]
ref_tm = test["fr"].tolist()                       # référence humaine si dispo, sinon proxy

rows = []
best_mt = HYPS[0]
for tau in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = top_score >= tau
    if keep.sum() < 3: continue
    row = {"seuil de similarité": tau, "couverture": round(float(keep.mean()), 2),
           "chrF++ mémoire": round(chrfpp.corpus_score([tm_fr[i] for i in np.where(keep)[0]], [[ref_tm[i] for i in np.where(keep)[0]]]).score, 1)}
    rows.append(row)
res7 = pd.DataFrame(rows); display(res7)
mt_ref = chrfpp.corpus_score(test[best_mt].tolist(), [ref_tm]).score if HAS_REF else np.nan
print("chrF++ du MT", best_mt[5:], "sur le même test :", "n/a (pas de référence humaine)" if not HAS_REF else round(mt_ref, 1))

## 8. Rôle C : noms propres du lexique
S2 injecte dans le prompt les noms propres de noms_v1.csv trouvés dans le segment arabe par correspondance exacte. Si Whisper écrit « السيسى » au lieu de « السيسي », ou perd un mot d'un nom composé, ça échoue. L'idée est de comparer chaque fenêtre de 1 à 4 mots aux formes du lexique avec EmbeddingGemma.

Protocole : la vérité terrain est la correspondance exacte sur le texte propre, puis on ajoute du bruit et on compare précision et rappel de la correspondance exacte et de fenêtres + EmbeddingGemma. Limite : la vérité terrain vient elle-même d'une correspondance exacte, donc la méthode exacte est parfaite sans bruit par construction.

In [ ]:
lex = pd.read_csv(LEXIQUE_URL)
surf = []                                        # (index d'entrée, forme arabe)
for i, r in lex.iterrows():
    forms = [r.forme_arabe] + (re.split(r"\s*\|\s*", str(r.variantes)) if isinstance(r.variantes, str) else [])
    surf += [(i, clean_ar(f)) for f in forms if len(clean_ar(f)) >= 4]
surf_idx = np.array([i for i, _ in surf]); surf_txt = [f for _, f in surf]
print(len(lex), "entrées,", len(surf_txt), "formes de surface")

def exact_pred(text):
    return {i for i, f in surf if re.search(rf"(?<!\w){re.escape(f)}(?!\w)", text)}

gold = [exact_pred(t) for t in D.ar]
has_gold = [i for i, g in enumerate(gold) if g]
print(len(has_gold), "segments contiennent au moins un nom du lexique |", sum(len(g) for g in gold), "occurrences")

E_surf = embed(model, surf_txt, S_PROMPT)
def windows(text, nmax=4):
    w = text.split(); return [" ".join(w[i:i + n]) for n in range(1, nmax + 1) for i in range(len(w) - n + 1)]

def emb_pred(text, tau):
    wins = windows(text)
    if not wins: return set()
    sim = embed(model, wins, S_PROMPT, bs=64) @ E_surf.T
    best = sim.max(1); arg = sim.argmax(1)
    return {int(surf_idx[arg[k]]) for k in np.where(best >= tau)[0]}

def prf(preds, golds):
    tp = sum(len(p & g) for p, g in zip(preds, golds)); fp = sum(len(p - g) for p, g in zip(preds, golds)); fn = sum(len(g - p) for p, g in zip(preds, golds))
    return round(tp / (tp + fp), 3) if tp + fp else 0.0, round(tp / (tp + fn), 3) if tp + fn else 0.0

neg = random.Random(SEED).sample([i for i in range(len(D)) if i not in has_gold], min(30, len(D) - len(has_gold)))
eval_idx = has_gold + neg
rows = []
for p in [0.0, 0.1, 0.2, 0.3]:
    texts = [D.ar[i] if p == 0 else add_asr_noise(D.ar[i], p, SEED + i) for i in eval_idx]
    gs = [gold[i] for i in eval_idx]
    pe, re_ = prf([exact_pred(t) for t in texts], gs)
    rows.append({"bruit": p, "méthode": "exacte", "précision": pe, "rappel": re_})
    for tau in [0.75, 0.85]:
        pr, rc = prf([emb_pred(t, tau) for t in texts], gs)
        rows.append({"bruit": p, "méthode": f"EmbeddingGemma τ={tau}", "précision": pr, "rappel": rc})
res8 = pd.DataFrame(rows); display(res8)

## 9. Accord avec COMETKiwi (optionnel)
Mettre RUN_COMETKIWI = True. Il faut avoir accès au modèle Unbabel/wmt22-cometkiwi-da (licence acceptée avec le même compte Hugging Face) et installer unbabel-comet, qui peut changer des versions de bibliothèques. À lancer après toutes les autres cellules, ou dans un runtime neuf en rechargeant hypotheses.csv.

In [ ]:
res9 = pd.DataFrame()
if RUN_COMETKIWI:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], check=False)
    from comet import download_model, load_from_checkpoint
    kiwi = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
    rows = []
    for h in HYPS:
        out = kiwi.predict([{"src": a, "mt": m} for a, m in zip(D.ar, D[h])], batch_size=16, gpus=1 if device == "cuda" else 0)
        k = np.array(out.scores); e = QE[h]
        rho, _ = spearmanr(k, e)
        worst_k = set(np.argsort(k)[: int(0.2 * len(k))]); worst_e = set(np.argsort(e)[: int(0.2 * len(e))])
        rows.append({"système": h[5:], "Spearman(EmbeddingGemma, COMETKiwi)": round(float(rho), 3),
                     "recouvrement des 20 % rejetés": round(len(worst_k & worst_e) / len(worst_k), 3)})
    res9 = pd.DataFrame(rows); display(res9)
else:
    print("COMETKiwi désactivé (RUN_COMETKIWI = False).")

## 10. Sauvegarde

In [ ]:
with pd.ExcelWriter("resultats_embeddinggemma_ar_fr.xlsx") as w:
    for name, tab in [("mt_chrf", res_mt), ("D_retrieval_prompts", res1), ("D_dimensions", res2), ("D_bruit", res3), ("D_latence", res4),
                      ("D_comparaison", res5), ("A_QE", res6), ("A_filtre20", res6b), ("B_memoire", res7), ("C_lexique", res8), ("A_cometkiwi", res9)]:
        if len(tab): tab.to_excel(w, sheet_name=name, index=False)
    D.assign(**{"qe::" + h[5:]: QE[h] for h in HYPS}).to_excel(w, sheet_name="donnees", index=False)
print("Écrit : resultats_embeddinggemma_ar_fr.xlsx, hypotheses.csv, segments_pour_le_groupe.csv")
if IN_COLAB:
    from google.colab import files
    files.download("resultats_embeddinggemma_ar_fr.xlsx")

## Conclusion
- EmbeddingGemma ne traduit pas, donc il se compare à COMETKiwi (QE) et pas à Qwen3. Dans le tableau commun il apparaît comme filtre de qualité (section 6) et comme mémoire de traduction (section 7).
- Chiffres à reporter : AUC de détection des mauvaises traductions, corrélation avec chrF++, recouvrement avec COMETKiwi, latence d'un encodage, robustesse au bruit ASR.
- Si l'AUC est élevée et l'accord avec COMETKiwi bon, il pourrait remplacer COMETKiwi au filtre D4-7 (plus léger, plus rapide, sans dépendance comet) ou servir de garde-fou en direct.
- Limites : sans référence humaine les scores sont mesurés contre des sorties de MT (optimistes), un seul épisode, vérité terrain de la section 8 issue d'une correspondance exacte, mémoire de traduction dépendante de la base.